In [1]:
!pip install -q vllm httpx

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\Lenovo\\AppData\\Local\\Temp\\pip-install-qhvt5f5a\\vllm_91200c433ab64341a7ef2f3665c278e1\\vllm/model_executor/layers/mamba/ops/configs/selective_state_update/headdim=64,dstate=128,device_name=NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition,cache_dtype=float16.json'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, sys, time, json, random, asyncio, statistics, subprocess
import httpx

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
BASE_ARGS = ["--dtype", "half", "--max-model-len", "2048", "--gpu-memory-utilization", "0.85"]

QUESTIONS = [
    "Answer with one word: what colour is the sky on a clear day?",
    "What is 12 times 7? Reply with only the number.",
    "Write three sentences about why the ocean is salty.",
    "List five fruits, comma separated.",
    "Explain in detail how a refrigerator keeps food cold.",
    "Say hello.",
    "Give a short definition of photosynthesis.",
    "Write a short paragraph about the history of the bicycle.",
    "What is the capital of Japan? One word.",
    "Describe how rain forms, step by step.",
    "Translate 'good morning' into French.",
    "Write a haiku about coffee.",
    "Explain what a database index is and why it speeds up queries.",
    "Name the planet closest to the sun.",
    "Give three tips for writing clean Python code.",
    "Summarise the water cycle in two sentences.",
]

def start_vllm(extra_args=(), port=8000, timeout_s=900, base_args=None):
    log_path = f"vllm_{port}.log"
    log = open(log_path, "w")
    cmd = ["vllm", "serve", MODEL_ID, "--port", str(port), *(base_args or BASE_ARGS), *extra_args]
    print("starting:", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    url = f"http://127.0.0.1:{port}"
    t0 = time.perf_counter()
    while time.perf_counter() - t0 < timeout_s:
        if proc.poll() is not None:
            log.close()
            raise RuntimeError("vLLM exited:\n" + open(log_path).read()[-4000:])
        try:
            if httpx.get(url + "/health", timeout=2).status_code == 200:
                startup_s = time.perf_counter() - t0
                print(f"vLLM ready after {startup_s:.0f}s")
                for line in open(log_path):
                    if "KV cache size" in line:
                        print("  log:", line.strip()[-120:])
                return proc, url, startup_s
        except httpx.HTTPError:
            pass
        time.sleep(2)
    proc.terminate()
    raise TimeoutError("vLLM did not start in time; see " + log_path)

def stop_vllm(proc):
    proc.terminate()
    try:
        proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        proc.kill()
    print("vLLM stopped")

def fetch_metrics(url):
    out = {}
    for line in httpx.get(url + "/metrics", timeout=5).text.splitlines():
        if not line.startswith("vllm:"):
            continue
        name_and_labels, value = line.rsplit(" ", 1)
        name = name_and_labels.split("{")[0]
        try:
            out[name] = out.get(name, 0.0) + float(value)
        except ValueError:
            pass
    return out

def metric(m, base):
    return m.get(base, m.get(base + "_total", 0.0))

async def stream_chat(client, url, question, max_tokens=48, system=None, stop_after_chunks=None):
    messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": question}]
    body = {"model": MODEL_ID, "messages": messages, "max_tokens": max_tokens, "temperature": 0,
            "stream": True, "stream_options": {"include_usage": True}}
    t0 = time.perf_counter()
    first, n_usage, chunks = None, None, 0
    async with client.stream("POST", url + "/v1/chat/completions", json=body) as r:
        r.raise_for_status()
        async for line in r.aiter_lines():
            if not line.startswith("data: "):
                continue
            data = line[6:]
            if data == "[DONE]":
                break
            msg = json.loads(data)
            if msg.get("usage"):
                n_usage = msg["usage"]["completion_tokens"]
            for ch in msg.get("choices", []):
                if ch.get("delta", {}).get("content"):
                    first = first or time.perf_counter()
                    chunks += 1
            if stop_after_chunks and chunks >= stop_after_chunks:
                break                                  # leaving the 'with' block closes the connection
    end = time.perf_counter()
    n = n_usage if n_usage is not None else chunks
    return {"ttft_s": (first or end) - t0, "e2e_s": end - t0, "tokens": n,
            "itl_s": (end - first) / (n - 1) if first and n > 1 else None}

def pct(values, q):
    v = sorted(values)
    return v[min(len(v) - 1, int(q * len(v)))] if v else float("nan")

def summarize(results, wall):
    itl = [x["itl_s"] for x in results if x["itl_s"] is not None]
    return {"n": len(results), "tok_per_s": sum(x["tokens"] for x in results) / wall,
            "ttft_p50": pct([x["ttft_s"] for x in results], .5), "ttft_p95": pct([x["ttft_s"] for x in results], .95),
            "e2e_p95": pct([x["e2e_s"] for x in results], .95),
            "itl_p50": pct(itl, .5), "itl_p95": pct(itl, .95)}

def show(label, s):
    print(f"{label:28s} n={s['n']:>3} | {s['tok_per_s']:7.1f} tok/s | TTFT p50 {s['ttft_p50']*1000:6.0f}ms "
          f"p95 {s['ttft_p95']*1000:6.0f}ms | ITL p50 {s['itl_p50']*1000:5.1f}ms p95 {s['itl_p95']*1000:5.1f}ms | "
          f"E2E p95 {s['e2e_p95']:5.2f}s")

class MetricsSampler:
    """Reads /metrics every `every` seconds in the background; keeps the highest values seen."""
    def __init__(self, url, every=0.25):
        self.url, self.every, self.samples, self._stop = url, every, [], asyncio.Event()
    async def _run(self):
        while not self._stop.is_set():
            try:
                self.samples.append((time.perf_counter(), await asyncio.to_thread(fetch_metrics, self.url)))
            except Exception:
                pass
            await asyncio.sleep(self.every)
    def start(self):
        self.task = asyncio.create_task(self._run())
    async def stop(self):
        self._stop.set()
        await self.task
    def peak(self, base):
        return max((metric(m, base) for _, m in self.samples), default=0.0)

def delta(before, after, base):
    return metric(after, base) - metric(before, base)

print("helpers ready")

helpers ready


#### Long prompts slowing down everyone else

In [3]:
async def closed_loop(url, concurrency, n_requests, max_tokens=48, system=None):
    todo, results = list(range(n_requests)), []
    async def user(client):
        while todo:
            q = QUESTIONS[todo.pop(0) % len(QUESTIONS)]
            results.append(await stream_chat(client, url, q, max_tokens=max_tokens, system=system))
    async with httpx.AsyncClient(timeout=600) as client:
        t0 = time.perf_counter()
        await asyncio.gather(*[user(client) for _ in range(concurrency)])
        return results, time.perf_counter() - t0

LONG_DOC = ("The history of computing spans calculating devices, mechanical machines, electronic "
            "computers and modern processors. ") * 90          # roughly 1,500 tokens

async def mixed_workload(url, short_users=8, n_short=48, n_long=6, long_gap_s=0.6):
    """8 users ask short questions while a few huge-prompt requests arrive in between."""
    short_results, long_results = [], []
    async with httpx.AsyncClient(timeout=600) as client:
        async def short_users_task():
            todo = list(range(n_short))
            async def user():
                while todo:
                    q = QUESTIONS[todo.pop(0) % len(QUESTIONS)]
                    short_results.append(await stream_chat(client, url, q, max_tokens=48))
            await asyncio.gather(*[user() for _ in range(short_users)])
        async def long_requests_task():
            for _ in range(n_long):
                await asyncio.sleep(long_gap_s)
                long_results.append(await stream_chat(
                    client, url, LONG_DOC + "\nSummarise the text above in one sentence.", max_tokens=32))
        t0 = time.perf_counter()
        await asyncio.gather(short_users_task(), long_requests_task())
        wall = time.perf_counter() - t0
    return summarize(short_results, wall), summarize(long_results, wall)

lab_a = {}
for label, extra in [("default (2048 tokens/step)", []),
                     ("max-num-batched-tokens=256", ["--max-num-batched-tokens", "256"])]:
    proc, url, _ = start_vllm(extra)
    try:
        await closed_loop(url, 4, 8)                                     # warm-up
        s_short, s_long = await mixed_workload(url)
        lab_a[label] = (s_short, s_long)
        show(f"{label} SHORT", s_short)
        show(f"{label} LONG", s_long)
    finally:
        stop_vllm(proc)   

starting: vllm serve Qwen/Qwen2.5-0.5B-Instruct --port 8000 --dtype half --max-model-len 2048 --gpu-memory-utilization 0.85


FileNotFoundError: [WinError 2] The system cannot find the file specified

#### Running out of KV cache, which causes preemption

In [ ]:
proc, url, _ = start_vllm(["--num-gpu-blocks-override", "128"],
                          base_args=["--dtype", "half", "--max-model-len", "1024"])
try:
    sampler = MetricsSampler(url)
    before = fetch_metrics(url)
    sampler.start()
    results, wall = await closed_loop(url, 16, 32, max_tokens=400)
    await sampler.stop()
    after = fetch_metrics(url)
    show("tiny KV cache, 16 users", summarize(results, wall))
    print(f"peak KV usage      : {sampler.peak('vllm:kv_cache_usage_perc'):.0%}")
    print(f"peak running       : {sampler.peak('vllm:num_requests_running'):.0f}")
    print(f"peak waiting       : {sampler.peak('vllm:num_requests_waiting'):.0f}")
    print(f"preemptions        : {delta(before, after, 'vllm:num_preemptions'):.0f}")
finally:
    stop_vllm(proc)